# Restart an Interrupted Extraction

A batch extraction over a large corpus can run for hours. If the process stops part way, the Amazon Bedrock batch jobs it submitted keep running, and their output lands in Amazon S3. Passing a `run_id` makes a run restartable: run it again with the same `run_id` and it takes up where the earlier run stopped.

This notebook starts a batch extraction, stops it while its Bedrock jobs are queued, shows what the run left in S3, and restarts it. It then shows a restart that is refused, builds a graph from the staged documents, and runs a query.

See [Restarting an interrupted extraction](https://awslabs.github.io/graphrag-toolkit/lexical-graph/indexing/#restarting-an-interrupted-extraction). Restart covers batch extraction with a fixed batch size. A run with a `run_id` and `auto_tune=True` raises a `ValueError`.

## Setup

If you haven't already, install the toolkit and dependencies using the [Setup](./00-Setup.ipynb) notebook.

This notebook uses [batch extraction](https://awslabs.github.io/graphrag-toolkit/lexical-graph/batch-extraction/), which the lexical-graph CloudFormation templates do not set up on their own. Deploy [`graphrag-toolkit-bedrock-batch.json`](https://github.com/awslabs/graphrag-toolkit/tree/main/examples/lexical-graph/cloudformation-templates/graphrag-toolkit-bedrock-batch.json) in the same Region first. It creates a bucket for batch input and output, a role Bedrock assumes to run the jobs, and a policy that lets the notebook role submit, check and stop those jobs.

Then create or update the lexical-graph stack with the `BatchClientPolicyArn` output in its `IamPolicyArn` parameter, and add the other two outputs to your `.env` file:

```
export EXTRACTION_BUCKET=<BatchBucket output>
export BATCH_INFERENCE_ROLE=<BatchInferenceRole output>
```

The extraction model, `EXTRACTION_MODEL`, must support [Bedrock batch inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-inference-supported.html).


### Dataset

Bedrock batch inference needs at least 100 records in a job. With fewer, the toolkit extracts chunk by chunk instead, and there is no batch job to restart. This notebook uses the 100 aviation accident reports in [`examples/lexical-graph/workshop/assets/source-data/ntsb-md`](https://github.com/awslabs/graphrag-toolkit/tree/main/examples/lexical-graph/workshop/assets/source-data/ntsb-md), taken from the [NTSB aviation investigation reports](https://www.ntsb.gov/investigations/AccidentReports/Pages/aviation.aspx). With the default chunking they split into about 1,100 chunks, so each of the two workers below submits one job of more than 500 records per extraction stage.

The next cell downloads the reports into `./ntsb-md`.

In [ ]:
%%bash
if [ ! -d ntsb-md ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/awslabs/graphrag-toolkit.git graphrag-toolkit-src
  git -C graphrag-toolkit-src sparse-checkout set examples/lexical-graph/workshop/assets/source-data/ntsb-md
  cp -r graphrag-toolkit-src/examples/lexical-graph/workshop/assets/source-data/ntsb-md ./ntsb-md
fi
ls ntsb-md | wc -l

## The extraction

The run is stopped by killing its process, so the first run has to run in a process of its own. The next cell writes the extraction to a file that both runs use. The first run and the restart then have the same settings, and the restart is not refused for a setting that differs by accident.

Each report is read as one document whose only metadata is its file name. A document's id is derived from its text and metadata, and a restart is refused when the ids differ from the plan, so the metadata must not change between runs. File dates, for example, would change if the reports were downloaded again.

`run_plan_store.staging_handler(run_id)` returns an `S3BasedDocs` handler for the collection that skips every source document already stored whole. On a first run it skips nothing.

In [ ]:
%%writefile restart_extraction.py
import os

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, GraphRAGConfig, IndexingConfig, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.extract import BatchConfig, RunPlanStore

from llama_index.core.schema import Document

DOCS_DIR = 'ntsb-md'
KEY_PREFIX = 'restart-example'
EXTRACTION_BATCH_SIZE = 100


def run_plan_store(collection_id):
    return RunPlanStore(
        bucket_name=os.environ['EXTRACTION_BUCKET'],
        key_prefix=f'{KEY_PREFIX}/extracted',
        collection_id=collection_id
    )


def read_report(file_name):
    with open(os.path.join(DOCS_DIR, file_name)) as f:
        return Document(text=f.read(), metadata={'file_name': file_name})


def load_docs():
    return [read_report(name) for name in sorted(os.listdir(DOCS_DIR)) if name.endswith('.md')]


def extract(collection_id, run_id, batch_size=EXTRACTION_BATCH_SIZE):

    set_logging_config('INFO')

    GraphRAGConfig.extraction_batch_size = batch_size
    GraphRAGConfig.extraction_num_workers = 2

    store = run_plan_store(collection_id)

    indexing_config = IndexingConfig(
        batch_config=BatchConfig(
            region=GraphRAGConfig.aws_region,
            bucket_name=os.environ['EXTRACTION_BUCKET'],
            key_prefix=f'{KEY_PREFIX}/batch-inference',
            role_arn=os.environ['BATCH_INFERENCE_ROLE'],
            max_batch_size=1000,
            max_num_concurrent_batches=1
        )
    )

    with (
        GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE']) as graph_store,
        VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE']) as vector_store
    ):

        graph_index = LexicalGraphIndex(
            graph_store,
            vector_store,
            indexing_config=indexing_config
        )

        graph_index.extract(
            load_docs(),
            handler=store.staging_handler(run_id),
            show_progress=True,
            run_id=run_id,
            run_plan_store=store
        )

The next cell picks a collection id and a run id for this notebook. Keep the run id: a restart is a run with the same `run_id`.

In [ ]:
%reload_ext dotenv
%dotenv

import os
import uuid

from graphrag_toolkit.lexical_graph import GraphRAGConfig, set_logging_config
from graphrag_toolkit.lexical_graph.indexing.extract.batch_extractor_base import LIVE_JOB_STATES

from restart_extraction import KEY_PREFIX, extract, run_plan_store

set_logging_config('INFO')

bucket_name = os.environ['EXTRACTION_BUCKET']
collection_id = f'ntsb-{uuid.uuid4().hex[:8]}'
run_id = f'run-{uuid.uuid4().hex[:8]}'

store = run_plan_store(collection_id)
manifest_store = store.manifest_store(run_id)


def jobs_recorded():
    records = manifest_store.read_partitions(GraphRAGConfig.s3)
    return {partition: record for partition, record in records.items() if record.job_arn}


def job_status(record):
    return GraphRAGConfig.bedrock.get_model_invocation_job(jobIdentifier=record.job_arn)['status']


print(f'collection_id: {collection_id}')
print(f'run_id       : {run_id}')

## Start a run and stop it

The next cell starts the first run in a child process with a session of its own, so that killing the session also kills the extraction workers. It waits until a partition record names a submitted Bedrock job, gives the other worker a minute to submit its job, and then kills the process. The child's output goes to `first-run.log`.

Killing the process does not stop the Bedrock jobs. They stay queued or running in Bedrock with nothing waiting on them.

In [ ]:
import signal
import subprocess
import sys
import time

with open('first-run.log', 'w') as log:
    first_run = subprocess.Popen(
        [
            sys.executable, '-c',
            'import sys; from restart_extraction import extract; extract(sys.argv[1], sys.argv[2])',
            collection_id, run_id
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

try:
    while not jobs_recorded():
        if first_run.poll() is not None:
            raise RuntimeError(f'The first run ended before it submitted a job. See first-run.log [exit code: {first_run.returncode}]')
        time.sleep(10)
    time.sleep(60)
finally:
    if first_run.poll() is None:
        os.killpg(first_run.pid, signal.SIGKILL)
    first_run.wait()

print(f'First run stopped [exit code: {first_run.returncode}]')

## What the stopped run left in S3

The run keeps its records under `s3://<bucket>/restart-example/extracted/<collection_id>/_runs/<run_id>/`:

- `plan.json`, written before any job was submitted: the document ids in order, the batch size and the worker count. A restart divides the documents the same way, so it produces the same partitions.
- `partitions/<partition_id>.json`, one per partition, naming the Bedrock job it submitted and the attempt number.

Staged documents sit beside the run directory, one prefix per source document, each with a completion marker under `_markers/`. Documents are staged after both extraction stages finish, so a run stopped during its first batch jobs has staged nothing yet and there are no markers.

`plan_resume` reads these records and the collection and reports what a restart will reuse. It submits nothing.

In [ ]:
from graphrag_toolkit.lexical_graph.indexing.extract.resume import plan_resume
from graphrag_toolkit.lexical_graph.indexing.load.s3_based_docs import is_completion_marker

plan = store.read(run_id, GraphRAGConfig.s3)
print(f'Plan: {len(plan.document_ids)} documents, batch_size: {plan.batch_size}, num_workers: {plan.num_workers}')
print()

interrupted = jobs_recorded()
for partition, record in interrupted.items():
    print(f'Partition {partition[:10]}  attempt: {record.attempt}  job: {record.job_name}  Bedrock status: {job_status(record)}')
print()

pages = GraphRAGConfig.s3.get_paginator('list_objects_v2').paginate(
    Bucket=bucket_name,
    Prefix=f'{store.key_prefix}/{collection_id}/'
)
keys = [obj['Key'] for page in pages for obj in page.get('Contents', [])]
for key in keys:
    print(key)
print(f'Completion markers: {len([key for key in keys if is_completion_marker(key)])}')
print()

report = plan_resume(manifest_store, GraphRAGConfig.s3)
print(report.describe())

## Restart with the same run id

The restart reads the plan and the partition records. For each partition with a recorded job, it asks Bedrock for the job's status:

- A job still queued or running is waited on, rather than submitted again.
- A completed job has its output downloaded from S3.
- A job that failed, stopped or expired is submitted again as the next attempt.

Partitions the first run never reached, such as those of the second extraction stage, are submitted as usual. The cell runs until every job has finished, which depends on how long Bedrock takes to schedule and run them.

Afterwards, a partition whose recorded job completed still names that job and is still on attempt 1. When the run finishes, its partition records are folded into `rollup.json` in the run directory.

In [ ]:
extract(collection_id, run_id)

for partition, record in manifest_store.read_partitions(GraphRAGConfig.s3).items():
    before = interrupted.get(partition)
    if before is None:
        submitted_by = 'submitted by the restart'
    elif before.job_arn == record.job_arn:
        submitted_by = 'same job as the first run'
    else:
        submitted_by = 'submitted again by the restart'
    print(f'Partition {partition[:10]}  attempt: {record.attempt}  state: {record.state}  job: {record.job_name}  ({submitted_by})')

## A changed setting is a new run

A restart has to divide the documents as the plan recorded. If `extraction_batch_size` or the set of documents differs, the restart is refused with a `RunPlanMismatch` before any work is submitted. To run with different settings, use a new `run_id`.

In [ ]:
from graphrag_toolkit.lexical_graph.indexing.extract import RunPlanMismatch

try:
    extract(collection_id, run_id, batch_size=50)
except RunPlanMismatch as e:
    print(f'Refused: {e}')

## Build and query

The build stage reads the staged collection like any other `S3BasedDocs` collection, and skips the run directory. The graph is built under the `restart` tenant so that it stays apart from the graphs the other notebooks build. See [Multi-Tenancy](./04-Multi-Tenancy.ipynb).

In [ ]:
from graphrag_toolkit.lexical_graph import LexicalGraphIndex, LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import S3BasedDocs

docs = S3BasedDocs(
    region=GraphRAGConfig.aws_region,
    bucket_name=bucket_name,
    key_prefix=store.key_prefix,
    collection_id=collection_id
)

with (
    GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE']) as graph_store,
    VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE']) as vector_store
):

    graph_index = LexicalGraphIndex(
        graph_store,
        vector_store,
        tenant_id='restart'
    )

    graph_index.build(docs, show_progress=True)

    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
        graph_store,
        vector_store,
        tenant_id='restart'
    )

    response = query_engine.query('What happened to the Bell 206 helicopter near Blair, Nebraska?')

print(response.response)

## Clean up

The next cell stops any Bedrock job from this run that is still queued or running, then deletes the staged collection, the run records and the batch inference files under `restart-example/` in the bucket. It does not remove the `restart` tenant's data from the graph and vector stores, or the local files this notebook created: `ntsb-md`, `graphrag-toolkit-src`, `restart_extraction.py` and `first-run.log`.

In [ ]:
for record in manifest_store.read_partitions(GraphRAGConfig.s3).values():
    if record.job_arn and job_status(record) in LIVE_JOB_STATES:
        GraphRAGConfig.bedrock.stop_model_invocation_job(jobIdentifier=record.job_arn)
        print(f'Stopped {record.job_name}')

for prefix in [f'{store.key_prefix}/{collection_id}/', f'{KEY_PREFIX}/batch-inference/']:
    pages = GraphRAGConfig.s3.get_paginator('list_objects_v2').paginate(Bucket=bucket_name, Prefix=prefix)
    keys = [obj['Key'] for page in pages for obj in page.get('Contents', [])]
    for i in range(0, len(keys), 1000):
        GraphRAGConfig.s3.delete_objects(
            Bucket=bucket_name,
            Delete={'Objects': [{'Key': key} for key in keys[i:i + 1000]]}
        )
    print(f'Deleted {len(keys)} objects under s3://{bucket_name}/{prefix}')